<a href="https://colab.research.google.com/github/sina-04/dental-yolo26-detection/blob/main/notebooks/dental_yolo26_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dental YOLO26 — reproducible T4 workflow

This staged notebook uses only Dental X-Ray Panoramic Dataset v6. Preparation reconstructs leakage-safe splits, creates isolated base and augmented training views, and fingerprints the exact data contract. Training is blocked until all 31 annotation contact sheets are reviewed. Two controlled 100-epoch YOLO26s experiments are selected using validation data; the held-out test set is evaluated in a separate step.

In [ ]:
from google.colab import drive
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU, then reconnect.'
print(torch.cuda.get_device_name(0))
print(f'{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB VRAM')


In [ ]:
from pathlib import Path

repo = Path('/content/dental-yolo26-detection')
if (repo / '.git').exists():
    !git -C {repo} pull --ff-only
else:
    !git clone https://github.com/sina-04/dental-yolo26-detection.git {repo}
%cd /content/dental-yolo26-detection
!python -m pip install -q -r requirements-colab.txt
!python -m unittest discover -s tests -v


## 1. Prepare and verify

The exact Kaggle v6 export is downloaded to ephemeral SSD. Raw images are not copied to Drive. Remove `--rebuild-data` on later runs to reuse an already prepared dataset in the same Colab session.

In [ ]:
!python -m src.colab_workflow \
  --stage prepare \
  --data-root /content/dental_yolo26_data \
  --rebuild-data


## 2. Required annotation audit

Inspect every sheet below. Red boxes are the target class; blue boxes are other labeled findings. Training cannot start without an approval tied to this dataset fingerprint. Approval records review completion—not clinical validation.

In [ ]:
from IPython.display import Image, display
from pathlib import Path

sheets = sorted(Path('reports/annotation_audit').glob('class_*.jpg'))
assert len(sheets) == 31, f'Expected 31 sheets, found {len(sheets)}'
for sheet in sheets:
    print(sheet.name)
    display(Image(filename=str(sheet), width=740))


In [ ]:
import subprocess, sys

AUDIT_APPROVED = False  # Set True only after reviewing all 31 sheets above.
REVIEWER = 'Your name'
NOTES = 'Summarize label quality findings and any questionable classes.'
assert AUDIT_APPROVED, 'Review every contact sheet, complete REVIEWER/NOTES, then set AUDIT_APPROVED=True.'
subprocess.run([sys.executable, '-m', 'src.approve_audit', '--reviewer', REVIEWER, '--notes', NOTES], check=True)


## 3. Train and select on validation data

The T4 profile runs two controlled YOLO26s experiments at 640 px and batch 16 for up to 100 epochs each: an original-image baseline and a medically constrained augmented view. All model settings are identical except the training view. Checkpoints are saved every epoch to the versioned Drive directory and incomplete runs resume from `last.pt`.

In [ ]:
!python -m src.colab_workflow \
  --stage train \
  --profile configs/colab_t4.yaml \
  --results-root /content/drive/MyDrive/dental-yolo26-detection/panoramic31-yolo26s-t4-v1


## 4. One-time held-out test evaluation

Run only after the validation-selected experiment is final. A matching completed result is reused. Do not pass `--force-test` merely to inspect or tune against test results.

In [ ]:
!python -m src.colab_workflow \
  --stage test \
  --profile configs/colab_t4.yaml \
  --results-root /content/drive/MyDrive/dental-yolo26-detection/panoramic31-yolo26s-t4-v1


## 5. Generate deliverable reports

In [ ]:
!python -m src.colab_workflow \
  --stage report \
  --results-root /content/drive/MyDrive/dental-yolo26-detection/panoramic31-yolo26s-t4-v1

results = Path('/content/drive/MyDrive/dental-yolo26-detection/panoramic31-yolo26s-t4-v1')
print('Best model:', results / 'artifacts/best.pt')
print('Metrics:', results / 'reports/final_metrics.json')
print('Report:', results / 'FINAL_REPORT.md')
print('Progress:', results / 'PROGRESS.md')
assert (results / 'artifacts/best.pt').exists()
assert (results / 'reports/final_metrics.json').exists()
